In [19]:
import os
import json
import hashlib
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from umap import UMAP
import hdbscan
from sklearn.cluster import AgglomerativeClustering
from anytree import Node, RenderTree

warnings.filterwarnings("ignore")

In [20]:
df = pd.read_json("D:/git/Taxonomy_Buidling_Textual_Corpora/data/icecat_data_train.json")

# Work only with first 5000 rows from the beginning
df = df.iloc[:5000].copy()

print("Raw sample size:", len(df))


Raw sample size: 5000


In [21]:
def split_path(path):
    if not isinstance(path, str):
        return []
    parts = [p.strip() for p in path.split(">") if p.strip()]
    return parts

df["num_levels"] = df["pathlist_names"].apply(lambda x: len(split_path(x)))

print("Counts BEFORE cleaning:")
print(df["num_levels"].value_counts().sort_index())

Counts BEFORE cleaning:
num_levels
3    2883
4    2117
Name: count, dtype: int64


In [22]:
import pandas as pd

def make_3_and_4(path):
    parts = split_path(path)

    # default
    path_3 = None
    level_4 = None

    if len(parts) >= 3:
        path_3 = " > ".join(parts[:3])   # A > B > C
    if len(parts) >= 4:
        level_4 = parts[3]               # D (4th level)

    return pd.Series({"path_3": path_3, "level_4": level_4})

df[["path_3", "level_4"]] = df["pathlist_names"].apply(make_3_and_4)

print(df[["pathlist_names", "path_3", "level_4"]].head(10))


                                            pathlist_names  \
1072689  Computers & Electronics>Computers>PCs/Workstat...   
906402   Computers & Electronics>Computers>Notebook Par...   
411281   Computers & Electronics>Computer Cables>Fibre ...   
425903   Computers & Electronics>Computers>Handheld Mob...   
1047582  Computers & Electronics>Computers>PCs/Workstat...   
904910   Computers & Electronics>Computers>Notebook Par...   
157385   Computers & Electronics>Software>Software Lice...   
934548   Computers & Electronics>Computers>Notebook Par...   
876762   Computers & Electronics>Computers>Notebook Par...   
397028   Computers & Electronics>Telecom & Navigation>M...   

                                                    path_3  \
1072689  Computers & Electronics > Computers > PCs/Work...   
906402   Computers & Electronics > Computers > Notebook...   
411281   Computers & Electronics > Computer Cables > Fi...   
425903   Computers & Electronics > Computers > Handheld...   
1047582

In [23]:
print("Rows:", len(df))
print("Unique num_levels (original):")
print(df["num_levels"].value_counts().sort_index())

print("\nCheck how many have a 4th level stored:")
print(df["level_4"].notna().sum(), "rows with level_4")


Rows: 5000
Unique num_levels (original):
num_levels
3    2883
4    2117
Name: count, dtype: int64

Check how many have a 4th level stored:
2117 rows with level_4


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5000 entries, 1072689 to 1121736
Data columns (total 48 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   Brand                                       5000 non-null   object 
 1   BrandInfo.BrandLocalName                    5000 non-null   object 
 2   BrandInfo.BrandLogo                         4994 non-null   object 
 3   BrandInfo.BrandName                         5000 non-null   object 
 4   BrandLogo                                   4994 non-null   object 
 5   BrandPartCode                               5000 non-null   object 
 6   BulletPoints                                4654 non-null   object 
 7   Category.CategoryID                         5000 non-null   int64  
 8   Category.Name.Language                      5000 non-null   object 
 9   Category.Name.Value                         5000 non-null   object 
 10  Descript

In [25]:
# Extract A, B, C from path_3
df[["A", "B", "C"]] = (
    df["path_3"]
    .str.split(">", expand=True)
    .apply(lambda col: col.str.strip())
)


In [26]:
df.head()

,Brand,BrandInfo.BrandLocalName,BrandInfo.BrandLogo,BrandInfo.BrandName,BrandLogo,BrandPartCode,BulletPoints,Category.CategoryID,Category.Name.Language,Category.Name.Value,...,VirtualCategory,SummaryDescription,pathlist_ids,pathlist_names,num_levels,path_3,level_4,A,B,C
1072689,ASUS,,https://images.icecat.biz/img/brand/thumb/161_...,ASUS,https://images.icecat.biz/img/brand/thumb/161_...,K31CD-IT049T,[],153,EN,PCs/Workstations,...,"[{'VirtualCategoryID': 195, 'UNCATID': '431718...",NaN,2833>150>153,Computers & Electronics>Computers>PCs/Workstat...,3,Computers & Electronics > Computers > PCs/Work...,None,Computers & Electronics,Computers,PCs/Workstations
906402,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,686915-A41,[],2509,EN,Notebook Spare Parts,...,None,NaN,2833>150>8355>2509,Computers & Electronics>Computers>Notebook Par...,4,Computers & Electronics > Computers > Notebook...,Notebook Spare Parts,Computers & Electronics,Computers,Notebook Parts & Accessories
411281,C2G,,https://images.icecat.biz/img/brand/thumb/2834...,C2G,https://images.icecat.biz/img/brand/thumb/2834...,37745,[],953,EN,Fibre Optic Cables,...,None,NaN,2833>830>953,Computers & Electronics>Computer Cables>Fibre ...,3,Computers & Electronics > Computer Cables > Fi...,None,Computers & Electronics,Computer Cables,Fibre Optic Cables
425903,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,FA889AA#AC3,[],8194,EN,Handheld Mobile Computer Spare Parts,...,None,NaN,2833>150>8194,Computers & Electronics>Computers>Handheld Mob...,3,Computers & Electronics > Computers > Handheld...,None,Computers & Electronics,Computers,Handheld Mobile Computer Spare Parts
1047582,Lenovo,,https://images.icecat.biz/img/brand/thumb/728_...,Lenovo,https://images.icecat.biz/img/brand/thumb/728_...,109559U,[],153,EN,PCs/Workstations,...,"[{'VirtualCategoryID': 194, 'UNCATID': '431718...",NaN,2833>150>153,Computers & Electronics>Computers>PCs/Workstat...,3,Computers & Electronics > Computers > PCs/Work...,None,Computers & Electronics,Computers,PCs/Workstations


In [27]:
# Basic Stats
print("Total products:", len(df))
print("Unique A-level:", df["A"].nunique())
print("Unique B-level:", df["B"].nunique())
print("Unique C-level:", df["C"].nunique())


Total products: 5000
Unique A-level: 1
Unique B-level: 17
Unique C-level: 160


In [30]:
#Distribution of A / B / C/
print("\nProducts per A-level:")
print(df["A"].value_counts())

print("\nProducts per B-level (top 10):")
print(df["B"].value_counts().head(20))

print("\nProducts per C-level (top 10):")
print(df["C"].value_counts().head(20))

print("\nProducts per C-level (bottom 10):")
print(df["C"].value_counts().tail(10))



Products per A-level:
A
Computers & Electronics    5000
Name: count, dtype: int64

Products per B-level (top 10):
B
Computers                           2202
Printers & Scanners                  368
Computer Components                  330
Warranty & Support                   318
Software                             296
Data Storage                         284
TVs & Monitors                       251
Computer Cables                      205
Telecom & Navigation                 189
Batteries & Power Supplies           141
Data Input Devices                   111
Consumer Audio & Video Equipment     110
Projectors                            73
Photo & Video Equipment               66
Networking                            47
Office Electronics                     7
Smart Wearables                        2
Name: count, dtype: int64

Products per C-level (top 10):
C
Notebook Parts & Accessories     1019
Notebooks                         774
Warranty & Support Extensions     288
Data Storage

In [ ]:
# C-level Imbalance Quantiles
# (This shows how skewed leaf categories are)
prod_per_C = df["C"].value_counts()

print("\nC-level product quantiles:")
print(prod_per_C.quantile([0.1, 0.25, 0.5, 0.75, 0.9]))



C-level product quantiles:
0.10     1.0
0.25     2.0
0.50     5.0
0.75    14.0
0.90    60.3
Name: count, dtype: float64


In [32]:
# Basic counts
print("Total products:", len(df))
print("Unique A levels:", df["A"].nunique())
print("Unique B levels:", df["B"].nunique())
print("Unique C levels:", df["C"].nunique())

# Compute product counts
prod_A = df["A"].value_counts()
prod_B = df["B"].value_counts()
prod_C = df["C"].value_counts()


Total products: 5000
Unique A levels: 1
Unique B levels: 17
Unique C levels: 160


In [34]:
# Identify problematic A/B/C categories
# Very large categories (top 10)
print("\n🔥 TOP 10 LARGEST C categories:")
print(prod_C.head(10))

# Very small categories (bottom 10)
print("\n❗ BOTTOM 10 SMALLEST C categories:")
print(prod_C.tail(10))

# C categories with < 10 products
weak_C = prod_C[prod_C < 10]
print("\n❗ Weak C categories (<10 products):", len(weak_C))
print(weak_C.head(20))

# B categories with < 3 C children
C_children_per_B = df.groupby("B")["C"].nunique()
weak_B = C_children_per_B[C_children_per_B < 3]

print("\n❗ Weak B categories (<3 C children):", len(weak_B))
print(weak_B.head(20))



🔥 TOP 10 LARGEST C categories:
C
Notebook Parts & Accessories     1019
Notebooks                         774
Warranty & Support Extensions     288
Data Storage Devices              273
System Components                 241
PCs/Workstations                  227
Software Licenses/Upgrades        227
Printing Supplies                 215
Keyboards                          85
Chassis Components                 84
Name: count, dtype: int64

❗ BOTTOM 10 SMALLEST C categories:
C
SCART Cables                                         1
Fax Machines                                         1
Sport Watch Accessories                              1
Network Switch Components                            1
Uninterruptible Power Supplies (UPSs) Accessories    1
USB Graphics Adapters                                1
Two-Way Radios                                       1
UPS Battery Cabinets                                 1
AV Receivers                                         1
S-Video Cables             

In [35]:
#Identify dominant categories (too large)
# Very large C categories (skewing the distribution)
dominant_C = prod_C[prod_C > prod_C.quantile(0.95)]

print("\n🔥 Dominant C categories (>95th percentile):", len(dominant_C))
print(dominant_C)



🔥 Dominant C categories (>95th percentile): 8
C
Notebook Parts & Accessories     1019
Notebooks                         774
Warranty & Support Extensions     288
Data Storage Devices              273
System Components                 241
PCs/Workstations                  227
Software Licenses/Upgrades        227
Printing Supplies                 215
Name: count, dtype: int64


In [37]:
B_summary = (
    df.groupby(["A", "B"])
      .agg(num_products=("C", "count"),
           num_C=("C", "nunique"))
      .reset_index()
      .sort_values("num_products", ascending=False)
)

print(" B-level imbalance summary:")
print(B_summary.head(20))


 B-level imbalance summary:
                          A                                 B  num_products  \
3   Computers & Electronics                         Computers          2202   
10  Computers & Electronics               Printers & Scanners           368   
2   Computers & Electronics               Computer Components           330   
16  Computers & Electronics                Warranty & Support           318   
13  Computers & Electronics                          Software           296   
6   Computers & Electronics                      Data Storage           284   
14  Computers & Electronics                    TVs & Monitors           251   
1   Computers & Electronics                   Computer Cables           205   
15  Computers & Electronics              Telecom & Navigation           189   
0   Computers & Electronics        Batteries & Power Supplies           141   
5   Computers & Electronics                Data Input Devices           111   
4   Computers & Electron

In [38]:
def report_AB(df):
    grouped = (
        df.groupby(["A", "B"])
          .agg(
              num_C=("C", "nunique"),
              num_products=("C", "size")
          )
          .reset_index()
    )

    for a in grouped["A"].unique():
        print(f"\nA: {a}")
        print("="*80)
        temp = grouped[grouped["A"] == a]
        for _, row in temp.iterrows():
            print(f"  B: {row['B']}")
            print(f"    #C-level subcategories: {row['num_C']}")
            print(f"    #products (total under this B): {row['num_products']}")


In [39]:
report_AB(df)


A: Computers & Electronics
  B: Batteries & Power Supplies
    #C-level subcategories: 12
    #products (total under this B): 141
  B: Computer Cables
    #C-level subcategories: 23
    #products (total under this B): 205
  B: Computer Components
    #C-level subcategories: 3
    #products (total under this B): 330
  B: Computers
    #C-level subcategories: 18
    #products (total under this B): 2202
  B: Consumer Audio & Video Equipment
    #C-level subcategories: 26
    #products (total under this B): 110
  B: Data Input Devices
    #C-level subcategories: 3
    #products (total under this B): 111
  B: Data Storage
    #C-level subcategories: 2
    #products (total under this B): 284
  B: Networking
    #C-level subcategories: 13
    #products (total under this B): 47
  B: Office Electronics
    #C-level subcategories: 3
    #products (total under this B): 7
  B: Photo & Video Equipment
    #C-level subcategories: 3
    #products (total under this B): 66
  B: Printers & Scanners
   

In [40]:
def report_ABC(df):
    grouped = (
        df.groupby(["A", "B", "C"])
          .size()
          .reset_index(name="num_products")
    )

    for a in grouped["A"].unique():
        print(f"\nA: {a}")
        print("="*80)
        tempA = grouped[grouped["A"] == a]

        for b in tempA["B"].unique():
            print(f"  B: {b}")
            tempB = tempA[tempA["B"] == b]

            for _, row in tempB.iterrows():
                print(f"    C: {row['C']}  ({row['num_products']} products)")


In [41]:
report_ABC(df)



A: Computers & Electronics
  B: Batteries & Power Supplies
    C: Battery Chargers  (4 products)
    C: Household Batteries  (11 products)
    C: Notebook Power Tips  (2 products)
    C: Portable Device Management Carts & Cabinets  (1 products)
    C: Power Adapters & Inverters  (63 products)
    C: Power Banks  (4 products)
    C: Power Distribution Units (PDUs)  (11 products)
    C: Power Supply Units  (16 products)
    C: UPS Batteries  (3 products)
    C: UPS Battery Cabinets  (1 products)
    C: Uninterruptible Power Supplies (UPSs)  (24 products)
    C: Uninterruptible Power Supplies (UPSs) Accessories  (1 products)
  B: Computer Cables
    C: Audio Cables  (3 products)
    C: Cable Interface/Gender Adapters  (20 products)
    C: Cable Protectors  (2 products)
    C: Coaxial Cables  (4 products)
    C: DVI Cables  (5 products)
    C: DisplayPort Cables  (3 products)
    C: Fibre Optic Cables  (29 products)
    C: FireWire Cables  (1 products)
    C: HDMI Cables  (11 products)
  

In [42]:
def report_ABC_products(df):
    for a in df["A"].unique():
        print(f"\nA: {a}")
        print("="*80)
        dfA = df[df["A"] == a]

        for b in dfA["B"].unique():
            print(f"  B: {b}")
            dfB = dfA[dfA["B"] == b]

            for c in dfB["C"].unique():
                dfC = dfB[dfB["C"] == c]
                print(f"    C: {c}  ({len(dfC)} products)")
                
                # show top 10 product codes
                for code in dfC["BrandPartCode"].fillna("").head(10):
                    print(f"        • {code}")


In [43]:
report_ABC_products(df)



A: Computers & Electronics
  B: Computers
    C: PCs/Workstations  (227 products)
        • K31CD-IT049T
        • 109559U
        • UN42-M031M
        • 30AT0066MB
        • 6PV24EA
        • 7HX46UT
        • 10MV002XUS
        • AEGIS 3 VR7RD-037US
        • 2ED43PA-E223
        • 1HK88EA
    C: Notebook Parts & Accessories  (1019 products)
        • 686915-A41
        • 659501-BB1
        • FRU00HT024
        • 448002-001
        • 900818-061
        • 25212106
        • SIC1085866LCD0
        • P000453890
        • SIC1099842LCD0
        • 25207264
    C: Handheld Mobile Computer Spare Parts  (1 products)
        • FA889AA#AC3
    C: Notebooks  (774 products)
        • NX.MPGEH.057
        • GE72 2QC-209NL
        • NX.VGAEY.002
        • 9S7-179E22-062
        • X8M21EA
        • GX531GX-ES016T
        • 20EMA008AU
        • 3JX62EA
        • PT343E-0NC05GGR
        • UX310UF-FC002T
    C: Servers  (15 products)
        • 5463M2G
        • SUB12GE
        • VFY:R1007SC040IN
    

In [46]:
# Compute metrics
products_per_AB = df.groupby(["A", "B"]).size()
C_children_per_AB = df.groupby(["A", "B"])["C"].nunique()

# Combine
ab_summary = pd.DataFrame({
    "products_in_AB": products_per_AB,
    "n_C_children": C_children_per_AB
})

# Sort by product count descending (like your example)
ab_summary = ab_summary.sort_values("products_in_AB", ascending=False).head(10)

print(ab_summary)


                                                    products_in_AB  \
A                       B                                            
Computers & Electronics Computers                             2202   
                        Printers & Scanners                    368   
                        Computer Components                    330   
                        Warranty & Support                     318   
                        Software                               296   
                        Data Storage                           284   
                        TVs & Monitors                         251   
                        Computer Cables                        205   
                        Telecom & Navigation                   189   
                        Batteries & Power Supplies             141   

                                                    n_C_children  
A                       B                                         
Computers & Electronics C

In [48]:
prod_per_C = df["C"].value_counts()
prod_per_C


C
Notebook Parts & Accessories     1019
Notebooks                         774
Warranty & Support Extensions     288
Data Storage Devices              273
System Components                 241
                                 ... 
USB Graphics Adapters               1
Two-Way Radios                      1
UPS Battery Cabinets                1
AV Receivers                        1
S-Video Cables                      1
Name: count, Length: 160, dtype: int64

In [54]:
#Keep only C categories with > 10 products
valid_C = prod_per_C[prod_per_C >= 20].index
df = df[df["C"].isin(valid_C)].copy()


In [56]:
print("Remaining rows:", len(df))
print("Unique C after filtering:", df["C"].nunique())
print("Min products per C (should be >20):", df["C"].value_counts().min())


Remaining rows: 4382
Unique C after filtering: 32
Min products per C (should be >20): 20


In [57]:
#final A-B summary
def print_final_AB(df):
    grouped = (
        df.groupby(["A", "B"])
          .agg(
              num_C_under_B=("C", "nunique"),
              num_products_under_B=("C", "size")
          )
          .reset_index()
    )

    for a in grouped["A"].unique():
        print(f"\nA: {a}")
        print("="*80)
        temp = grouped[grouped["A"] == a]

        for _, row in temp.iterrows():
            print(f"  B: {row['B']}")
            print(f"    #C-level subcategories: {row['num_C_under_B']}")
            print(f"    #products (under this B): {row['num_products_under_B']}")


In [58]:
print_final_AB(df)



A: Computers & Electronics
  B: Batteries & Power Supplies
    #C-level subcategories: 2
    #products (under this B): 87
  B: Computer Cables
    #C-level subcategories: 3
    #products (under this B): 116
  B: Computer Components
    #C-level subcategories: 2
    #products (under this B): 325
  B: Computers
    #C-level subcategories: 6
    #products (under this B): 2144
  B: Consumer Audio & Video Equipment
    #C-level subcategories: 1
    #products (under this B): 23
  B: Data Input Devices
    #C-level subcategories: 1
    #products (under this B): 85
  B: Data Storage
    #C-level subcategories: 1
    #products (under this B): 273
  B: Photo & Video Equipment
    #C-level subcategories: 1
    #products (under this B): 39
  B: Printers & Scanners
    #C-level subcategories: 4
    #products (under this B): 363
  B: Projectors
    #C-level subcategories: 1
    #products (under this B): 25
  B: Software
    #C-level subcategories: 2
    #products (under this B): 280
  B: TVs & Moni

In [59]:
def print_final_ABC(df):
    grouped = (
        df.groupby(["A", "B", "C"])
          .size()
          .reset_index(name="num_products_under_C")
    )

    for a in grouped["A"].unique():
        print(f"\nA: {a}")
        print("="*80)
        tempA = grouped[grouped["A"] == a]

        for b in tempA["B"].unique():
            print(f"  B: {b}")
            tempB = tempA[tempA["B"] == b]

            for _, row in tempB.iterrows():
                print(f"    C: {row['C']}  ({row['num_products_under_C']} products)")
                
print_final_ABC(df)




A: Computers & Electronics
  B: Batteries & Power Supplies
    C: Power Adapters & Inverters  (63 products)
    C: Uninterruptible Power Supplies (UPSs)  (24 products)
  B: Computer Cables
    C: Cable Interface/Gender Adapters  (20 products)
    C: Fibre Optic Cables  (29 products)
    C: Networking Cables  (67 products)
  B: Computer Components
    C: Chassis Components  (84 products)
    C: System Components  (241 products)
  B: Computers
    C: All-in-One PCs/Workstations  (60 products)
    C: Notebook Parts & Accessories  (1019 products)
    C: Notebooks  (774 products)
    C: PCs/Workstations  (227 products)
    C: Tablet Cases  (25 products)
    C: Tablets  (39 products)
  B: Consumer Audio & Video Equipment
    C: Audio Equipment Parts & Accessories  (23 products)
  B: Data Input Devices
    C: Keyboards  (85 products)
  B: Data Storage
    C: Data Storage Devices  (273 products)
  B: Photo & Video Equipment
    C: Cameras & Camcorders  (39 products)
  B: Printers & Scanners
 

In [60]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4382 entries, 1072689 to 1121736
Data columns (total 51 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   Brand                                       4382 non-null   object 
 1   BrandInfo.BrandLocalName                    4382 non-null   object 
 2   BrandInfo.BrandLogo                         4379 non-null   object 
 3   BrandInfo.BrandName                         4382 non-null   object 
 4   BrandLogo                                   4379 non-null   object 
 5   BrandPartCode                               4382 non-null   object 
 6   BulletPoints                                4088 non-null   object 
 7   Category.CategoryID                         4382 non-null   int64  
 8   Category.Name.Language                      4382 non-null   object 
 9   Category.Name.Value                         4382 non-null   object 
 10  Descript

In [61]:
df.head()

,Brand,BrandInfo.BrandLocalName,BrandInfo.BrandLogo,BrandInfo.BrandName,BrandLogo,BrandPartCode,BulletPoints,Category.CategoryID,Category.Name.Language,Category.Name.Value,...,VirtualCategory,SummaryDescription,pathlist_ids,pathlist_names,num_levels,path_3,level_4,A,B,C
1072689,ASUS,,https://images.icecat.biz/img/brand/thumb/161_...,ASUS,https://images.icecat.biz/img/brand/thumb/161_...,K31CD-IT049T,[],153,EN,PCs/Workstations,...,"[{'VirtualCategoryID': 195, 'UNCATID': '431718...",NaN,2833>150>153,Computers & Electronics>Computers>PCs/Workstat...,3,Computers & Electronics > Computers > PCs/Work...,None,Computers & Electronics,Computers,PCs/Workstations
906402,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,686915-A41,[],2509,EN,Notebook Spare Parts,...,None,NaN,2833>150>8355>2509,Computers & Electronics>Computers>Notebook Par...,4,Computers & Electronics > Computers > Notebook...,Notebook Spare Parts,Computers & Electronics,Computers,Notebook Parts & Accessories
411281,C2G,,https://images.icecat.biz/img/brand/thumb/2834...,C2G,https://images.icecat.biz/img/brand/thumb/2834...,37745,[],953,EN,Fibre Optic Cables,...,None,NaN,2833>830>953,Computers & Electronics>Computer Cables>Fibre ...,3,Computers & Electronics > Computer Cables > Fi...,None,Computers & Electronics,Computer Cables,Fibre Optic Cables
1047582,Lenovo,,https://images.icecat.biz/img/brand/thumb/728_...,Lenovo,https://images.icecat.biz/img/brand/thumb/728_...,109559U,[],153,EN,PCs/Workstations,...,"[{'VirtualCategoryID': 194, 'UNCATID': '431718...",NaN,2833>150>153,Computers & Electronics>Computers>PCs/Workstat...,3,Computers & Electronics > Computers > PCs/Work...,None,Computers & Electronics,Computers,PCs/Workstations
904910,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,659501-BB1,[],2509,EN,Notebook Spare Parts,...,None,NaN,2833>150>8355>2509,Computers & Electronics>Computers>Notebook Par...,4,Computers & Electronics > Computers > Notebook...,Notebook Spare Parts,Computers & Electronics,Computers,Notebook Parts & Accessories


In [71]:
#columns to keep
cols_needed = [
    "Brand",
    "BrandPartCode",
    "ProductName",
    "Description.LongProductName",
    "SummaryDescription.LongSummaryDescription",
    "Description.LongDesc",
    "A", "B", "C",
    "path_3"   # <-- Needed for evaluation
]


In [72]:
df_clean = df[cols_needed].copy()


In [73]:
# Build raw metadata text

def build_metadata(row):
    parts = []

    if pd.notna(row["Brand"]):
        parts.append(row["Brand"])

    if pd.notna(row["BrandPartCode"]):
        parts.append(row["BrandPartCode"])
        
    if pd.notna(row["ProductName"]):
        parts.append(row["ProductName"])

    if pd.notna(row["Description.LongProductName"]):
        parts.append(row["Description.LongProductName"])

    if pd.notna(row["SummaryDescription.LongSummaryDescription"]):
        parts.append(row["SummaryDescription.LongSummaryDescription"])

    if pd.notna(row["Description.LongDesc"]):
        parts.append(row["Description.LongDesc"])

    return " ".join(parts)

df_clean["metadata_text"] = df_clean.apply(build_metadata, axis=1)


In [74]:
df_clean

,Brand,BrandPartCode,ProductName,Description.LongProductName,SummaryDescription.LongSummaryDescription,Description.LongDesc,A,B,C,path_3,metadata_text
1072689,ASUS,K31CD-IT049T,K31CD-IT049T,"Intel Core i7-6700 (8M Cache, 3.4GHz), 16GB RA...",ASUS K31CD-IT049T. Processor frequency: 3.4 GH...,<b>Smart Multimedia Performance</b><br>\nVivoP...,Computers & Electronics,Computers,PCs/Workstations,Computers & Electronics > Computers > PCs/Work...,ASUS K31CD-IT049T K31CD-IT049T Intel Core i7-6...
906402,HP,686915-A41,686915-A41,Keyboard in midnight black finish with backlig...,HP 686915-A41. Type: Keyboard. Keyboard langua...,,Computers & Electronics,Computers,Notebook Parts & Accessories,Computers & Electronics > Computers > Notebook...,HP 686915-A41 686915-A41 Keyboard in midnight ...
411281,C2G,37745,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,Get the performance you demand at a price that...,Computers & Electronics,Computer Cables,Fibre Optic Cables,Computers & Electronics > Computer Cables > Fi...,C2G 37745 1m ST/SC Plenum-Rated 9/125 Duplex S...
1047582,Lenovo,109559U,C30,"Intel Xeon E5-2620 (15M Cache, 2.00 GHz, 7.20 ...",Lenovo ThinkStation C30. Processor frequency: ...,The C30 builds on its award-winning design as ...,Computers & Electronics,Computers,PCs/Workstations,Computers & Electronics > Computers > PCs/Work...,Lenovo 109559U C30 Intel Xeon E5-2620 (15M Cac...
904910,HP,659501-BB1,659501-BB1,Keyboard in ash black for use in Israel (inclu...,HP 659501-BB1. Type: Keyboard. Keyboard langua...,,Computers & Electronics,Computers,Notebook Parts & Accessories,Computers & Electronics > Computers > Notebook...,HP 659501-BB1 659501-BB1 Keyboard in ash black...
...,...,...,...,...,...,...,...,...,...,...,...
231002,DELL,490-BDZR,490-BDZR,Radeon Pro WX 2100 2 GB 1 DP 2 mDP (Precision),"DELL 490-BDZR. Graphics processor family: AMD,...",The new Radeon™ Pro WX 2100 graphics card is t...,Computers & Electronics,Computer Components,System Components,Computers & Electronics > Computer Components ...,DELL 490-BDZR 490-BDZR Radeon Pro WX 2100 2 GB...
978436,Toshiba,SIC1094057LCD0,SIC1094057LCD0,Satellite A660 15.6 replacement laptop LCD screen,Toshiba SIC1094057LCD0. Type: Display. Display...,,Computers & Electronics,Computers,Notebook Parts & Accessories,Computers & Electronics > Computers > Notebook...,Toshiba SIC1094057LCD0 SIC1094057LCD0 Satellit...
1101608,HP,6GV21PA,Elite Slice G2 with Microsoft Teams Rooms,None,HP Elite Slice G2 with Microsoft Teams Rooms,None,Computers & Electronics,Computers,PCs/Workstations,Computers & Electronics > Computers > PCs/Work...,HP 6GV21PA Elite Slice G2 with Microsoft Teams...
66861,LG,T300.ADEUPK,T300,None,"LG T300. Display diagonal: 6.1 cm (2.4""), Disp...",None,Computers & Electronics,Telecom & Navigation,Smartphones,Computers & Electronics > Telecom & Navigation...,LG T300.ADEUPK T300 LG T300. Display diagonal:...


In [75]:
import re

def clean_text(t):
    if not isinstance(t, str):
        return ""

    t = re.sub(r"<[^>]+>", " ", t)          # remove HTML tags
    t = re.sub(r"\s+", " ", t)              # collapse spaces
    t = t.replace("\xa0", " ")              # remove non-breaking
    t = t.strip()

    return t

df_clean["metadata_text_clean"] = df_clean["metadata_text"].apply(clean_text)


In [76]:
df_clean.head()

,Brand,BrandPartCode,ProductName,Description.LongProductName,SummaryDescription.LongSummaryDescription,Description.LongDesc,A,B,C,path_3,metadata_text,metadata_text_clean
1072689,ASUS,K31CD-IT049T,K31CD-IT049T,"Intel Core i7-6700 (8M Cache, 3.4GHz), 16GB RA...",ASUS K31CD-IT049T. Processor frequency: 3.4 GH...,<b>Smart Multimedia Performance</b><br>\nVivoP...,Computers & Electronics,Computers,PCs/Workstations,Computers & Electronics > Computers > PCs/Work...,ASUS K31CD-IT049T K31CD-IT049T Intel Core i7-6...,ASUS K31CD-IT049T K31CD-IT049T Intel Core i7-6...
906402,HP,686915-A41,686915-A41,Keyboard in midnight black finish with backlig...,HP 686915-A41. Type: Keyboard. Keyboard langua...,,Computers & Electronics,Computers,Notebook Parts & Accessories,Computers & Electronics > Computers > Notebook...,HP 686915-A41 686915-A41 Keyboard in midnight ...,HP 686915-A41 686915-A41 Keyboard in midnight ...
411281,C2G,37745,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,Get the performance you demand at a price that...,Computers & Electronics,Computer Cables,Fibre Optic Cables,Computers & Electronics > Computer Cables > Fi...,C2G 37745 1m ST/SC Plenum-Rated 9/125 Duplex S...,C2G 37745 1m ST/SC Plenum-Rated 9/125 Duplex S...
1047582,Lenovo,109559U,C30,"Intel Xeon E5-2620 (15M Cache, 2.00 GHz, 7.20 ...",Lenovo ThinkStation C30. Processor frequency: ...,The C30 builds on its award-winning design as ...,Computers & Electronics,Computers,PCs/Workstations,Computers & Electronics > Computers > PCs/Work...,Lenovo 109559U C30 Intel Xeon E5-2620 (15M Cac...,Lenovo 109559U C30 Intel Xeon E5-2620 (15M Cac...
904910,HP,659501-BB1,659501-BB1,Keyboard in ash black for use in Israel (inclu...,HP 659501-BB1. Type: Keyboard. Keyboard langua...,,Computers & Electronics,Computers,Notebook Parts & Accessories,Computers & Electronics > Computers > Notebook...,HP 659501-BB1 659501-BB1 Keyboard in ash black...,HP 659501-BB1 659501-BB1 Keyboard in ash black...


In [77]:
from sentence_transformers import SentenceTransformer

texts = df_clean["metadata_text_clean"].fillna("").tolist()

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(
    texts,
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True
)


Batches:   0%|          | 0/18 [00:00<?, ?it/s]

In [78]:
from umap import UMAP

umap = UMAP(
    n_neighbors=15,
    min_dist=0.0,
    n_components=25,
    metric="cosine",
    random_state=42
)

reduced = umap.fit_transform(embeddings)


In [79]:
import hdbscan

clusterer_C = hdbscan.HDBSCAN(
    min_cluster_size=20,   # consistent with your min 20 products per C
    min_samples=1,
    metric="euclidean"
)

df_clean["C_id"] = clusterer_C.fit_predict(reduced)

print("All cluster IDs:", sorted(df_clean["C_id"].unique()))


All cluster IDs: [-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76]


In [80]:
df_clean.head()

,Brand,BrandPartCode,ProductName,Description.LongProductName,SummaryDescription.LongSummaryDescription,Description.LongDesc,A,B,C,path_3,metadata_text,metadata_text_clean,C_id
1072689,ASUS,K31CD-IT049T,K31CD-IT049T,"Intel Core i7-6700 (8M Cache, 3.4GHz), 16GB RA...",ASUS K31CD-IT049T. Processor frequency: 3.4 GH...,<b>Smart Multimedia Performance</b><br>\nVivoP...,Computers & Electronics,Computers,PCs/Workstations,Computers & Electronics > Computers > PCs/Work...,ASUS K31CD-IT049T K31CD-IT049T Intel Core i7-6...,ASUS K31CD-IT049T K31CD-IT049T Intel Core i7-6...,45
906402,HP,686915-A41,686915-A41,Keyboard in midnight black finish with backlig...,HP 686915-A41. Type: Keyboard. Keyboard langua...,,Computers & Electronics,Computers,Notebook Parts & Accessories,Computers & Electronics > Computers > Notebook...,HP 686915-A41 686915-A41 Keyboard in midnight ...,HP 686915-A41 686915-A41 Keyboard in midnight ...,75
411281,C2G,37745,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,Get the performance you demand at a price that...,Computers & Electronics,Computer Cables,Fibre Optic Cables,Computers & Electronics > Computer Cables > Fi...,C2G 37745 1m ST/SC Plenum-Rated 9/125 Duplex S...,C2G 37745 1m ST/SC Plenum-Rated 9/125 Duplex S...,33
1047582,Lenovo,109559U,C30,"Intel Xeon E5-2620 (15M Cache, 2.00 GHz, 7.20 ...",Lenovo ThinkStation C30. Processor frequency: ...,The C30 builds on its award-winning design as ...,Computers & Electronics,Computers,PCs/Workstations,Computers & Electronics > Computers > PCs/Work...,Lenovo 109559U C30 Intel Xeon E5-2620 (15M Cac...,Lenovo 109559U C30 Intel Xeon E5-2620 (15M Cac...,10
904910,HP,659501-BB1,659501-BB1,Keyboard in ash black for use in Israel (inclu...,HP 659501-BB1. Type: Keyboard. Keyboard langua...,,Computers & Electronics,Computers,Notebook Parts & Accessories,Computers & Electronics > Computers > Notebook...,HP 659501-BB1 659501-BB1 Keyboard in ash black...,HP 659501-BB1 659501-BB1 Keyboard in ash black...,75


In [81]:
cluster_counts = df_clean["C_id"].value_counts().sort_values(ascending=False)

print(cluster_counts)


C_id
 14    361
-1     212
 10    181
 69    162
 45    158
      ... 
 55     20
 12     20
 7      20
 46     20
 39     20
Name: count, Length: 78, dtype: int64


In [82]:
summary = (
    df_clean.groupby("C_id")
            .size()
            .reset_index(name="num_products")
            .sort_values("num_products", ascending=False)
)

print(summary)


    C_id  num_products
15    14           361
0     -1           212
11    10           181
70    69           162
46    45           158
..   ...           ...
8      7            20
44    43            20
47    46            20
13    12            20
56    55            20

[78 rows x 2 columns]


In [85]:
def make_evidence(subdf, max_chars=1000, max_items=30):
    items = []
    for txt in subdf["metadata_text_clean"].fillna("").tolist():
        t = txt.strip()
        if not t:
            continue
        items.append("- " + t[:200])
        if len(items) >= max_items:
            break

    blob = "\n".join(items)
    return blob[:max_chars]


In [86]:
import subprocess
import hashlib

def run_llm(prompt, model="llama3.2"):
    cmd = ["ollama", "run", model]
    result = subprocess.run(
        cmd,
        input=prompt.encode("utf-8"),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    stderr_text = result.stderr.decode("utf-8", errors="ignore")
    if stderr_text.strip():
        print("⚠ Ollama stderr:\n", stderr_text)
    stdout_text = result.stdout.decode("utf-8", errors="ignore")
    return stdout_text

label_cache = {}

def cached_llm_label(prompt, max_words=4, model="llama3.2"):
    key = hashlib.sha256(prompt.encode("utf-8")).hexdigest()
    if key in label_cache:
        return label_cache[key]

    raw = run_llm(prompt, model=model)
    label = " ".join(raw.strip().split()[:max_words])
    label_cache[key] = label
    return label


In [88]:
C_name_map = {}
unique_cids = sorted(df_clean["C_id"].unique())
print("All cluster IDs:", unique_cids)

for cid in unique_cids:
    if cid == -1:   # skip noise
        continue

    subdf = df_clean[df_clean["C_id"] == cid]
    blob = make_evidence(subdf, max_chars=1000, max_items=30)

    if not blob:
        print(f"C_id {cid} → EMPTY SKIP")
        continue

    prompt = f"""
[SYSTEM]
You are an expert in product taxonomy.
You receive a group of similar products and must assign a concise C-level category name.
The name MUST be:
- At most 4 words
- Generic (no brand names)
- In English
- Describing the type of product, not features.

[USER]
Here is a product group:

{blob}

Please respond with ONLY the category name. No explanation, no punctuation, no quotes.
"""

    label = cached_llm_label(prompt, max_words=4)
    C_name_map[cid] = label

    print(f"C_id {cid} → {label}")

df_clean["C_name"] = df_clean["C_id"].map(C_name_map)


All cluster IDs: [-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76]
⚠ Ollama stderr:
 ⠙ ⠹ ⠸ ⠼ ⠼ ⠦ 
C_id 0 → Laptop displays
⚠ Ollama stderr:
 ⠙ ⠙ ⠸ ⠸ 
C_id 1 → Laptop Displays
⚠ Ollama stderr:
 ⠋ ⠹ ⠹ ⠸ 
C_id 2 → Antivirus Software for Businesses
⚠ Ollama stderr:
 ⠙ ⠙ ⠸ ⠸ ⠴ 
C_id 3 → laptops
⚠ Ollama stderr:
 ⠋ ⠹ ⠸ ⠼ ⠴ 
C_id 4 → Laptop Computers
⚠ Ollama stderr:
 ⠋ ⠙ ⠹ ⠸ 
C_id 5 → Laptop display replacements
⚠ Ollama stderr:
 ⠋ ⠙ ⠹ ⠼ 
C_id 6 → Extended Warranty Services
⚠ Ollama stderr:
 ⠋ ⠹ ⠹ ⠼ 
C_id 7 → Hardware Maintenance and Repair
⚠ Ollama stderr:
 ⠙ ⠙ ⠹ ⠸ 
C_id 8 → Lenovo Notebooks Motherboards
⚠ Ollama stderr:
 ⠋ ⠙ ⠸ ⠼ 
C_id 9 → Endpoint Antivirus Software
⚠ Ollama stderr:
 ⠙ ⠹ ⠹ ⠼ ⠴ 
C_id 10 → Laptop/Desktop Computers
⚠ Ollama std

In [91]:
# Cluster C → B (B_id) and name B clusters
# Cluster C centroids into B clusters
from sklearn.cluster import AgglomerativeClustering
import numpy as np

# 1. Compute centroids for each C_id (exclude noise -1)
valid_cids = [c for c in df_clean["C_id"].unique() if c != -1]

C_centroids = []
C_labels = []

for cid in valid_cids:
    idx = df_clean["C_id"] == cid
    C_centroids.append(reduced[idx].mean(axis=0))   # 'reduced' = UMAP output
    C_labels.append(cid)

C_centroids = np.vstack(C_centroids)

# 2. Cluster centroids into B-level groups
agg_B = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=1.0
)
B_ids = agg_B.fit_predict(C_centroids)

# 3. Map C_id → B_id
C_to_B = dict(zip(C_labels, B_ids))
df_clean["B_id"] = df_clean["C_id"].map(lambda c: C_to_B.get(c, -1))

print(df_clean[["C_id", "C_name", "B_id"]].head())


         C_id                    C_name  B_id
1072689    45                   Laptops     3
906402     75        Portable Keyboards    20
411281     33        Fiber Optic Cables     8
1047582    10  Laptop/Desktop Computers    31
904910     75        Portable Keyboards    20


In [92]:
# Name each B_id using LLaMA (using its C_names)
def name_B_clusters(df):
    B_name_map = {}
    for bid in sorted(df["B_id"].unique()):
        if bid == -1:
            continue
        names = df.loc[df["B_id"] == bid, "C_name"].dropna().unique().tolist()
        sample_names = ", ".join(names[:10])

        prompt = f"""
[SYSTEM]
You are an expert in product taxonomy.
You see a list of C-level subcategories and must assign a concise B-level group name.
The name MUST be:
- At most 4 words
- Generic (no brand names)
- In English.

[USER]
Subcategories:
{sample_names}

Respond with ONLY the B-level category name."""
        label = cached_llm_label(prompt, max_words=4)
        B_name_map[bid] = label
        print(f"B_id {bid} → {label}")
    return B_name_map

B_name_map = name_B_clusters(df_clean)
df_clean["B_name"] = df_clean["B_id"].map(B_name_map)


⚠ Ollama stderr:
 ⠋ ⠹ ⠹ ⠼ ⠼ 
B_id 0 → Display Replacement Screens
⚠ Ollama stderr:
 ⠙ ⠹ ⠸ 
B_id 1 → Endpoint Security Solutions
⚠ Ollama stderr:
 ⠙ ⠙ ⠹ ⠸ 
B_id 2 → Service and Maintenance Options
⚠ Ollama stderr:
 ⠙ ⠙ ⠸ 
B_id 3 → Portable Personal Computers
⚠ Ollama stderr:
 ⠋ ⠹ ⠹ ⠼ 
B_id 4 → Post-Warranty Maintenance Plans
⚠ Ollama stderr:
 ⠙ ⠙ ⠸ ⠼ 
B_id 5 → Laptop Computer Components
⚠ Ollama stderr:
 ⠙ ⠹ ⠹ ⠼ 
B_id 6 → Product Maintenance Services
⚠ Ollama stderr:
 ⠋ ⠹ ⠹ ⠼ 
B_id 7 → Portable Computer Screens
⚠ Ollama stderr:
 ⠋ ⠹ ⠹ ⠼ 
B_id 8 → Network Cable Products
⚠ Ollama stderr:
 ⠋ ⠹ ⠸ 
B_id 9 → Portable Storage Devices
⚠ Ollama stderr:
 ⠋ ⠙ ⠸ ⠼ 
B_id 10 → Computer Input Devices
⚠ Ollama stderr:
 ⠋ ⠹ ⠹ ⠸ 
B_id 11 → Personal Computing Devices
⚠ Ollama stderr:
 ⠙ ⠹ 
B_id 12 → Desktop Computer Enclosures
⚠ Ollama stderr:
 ⠋ ⠹ ⠹ 
B_id 13 → Visual Display Systems
⚠ Ollama stderr:
 ⠋ ⠙ 
B_id 14 → Power Supplies
⚠ Ollama stderr:
 ⠙ ⠹ ⠹ 
B_id 15 → Electronics Peripherals
⚠ Ollama stderr:

In [ ]:
# 1. Compute centroids for each B_id
# Cluster B → A (A_id) and name A clusters
# Cluster B centroids into A groups
valid_bids = [b for b in df_clean["B_id"].unique() if b != -1]

B_centroids = []
B_labels = []

for bid in valid_bids:
    idx = df_clean["B_id"] == bid
    B_centroids.append(reduced[idx].mean(axis=0))
    B_labels.append(bid)

B_centroids = np.vstack(B_centroids)

# 2. Agglomerative clustering for A
agg_A = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=1.2
)
A_ids = agg_A.fit_predict(B_centroids)

# 3. Map B_id → A_id
B_to_A = dict(zip(B_labels, A_ids))
df_clean["A_id"] = df_clean["B_id"].map(lambda b: B_to_A.get(b, -1))

print(df_clean[["A_id", "B_id", "B_name"]].head())


         A_id  B_id                       B_name
1072689    31     3  Portable Personal Computers
906402     40    20         Mobile Input Devices
411281     35     8       Network Cable Products
1047582    29    31   Personal Computing Devices
904910     40    20         Mobile Input Devices


In [94]:
def name_A_clusters(df):
    A_name_map = {}
    for aid in sorted(df["A_id"].unique()):
        if aid == -1:
            continue
        names = df.loc[df["A_id"] == aid, "B_name"].dropna().unique().tolist()
        sample_names = ", ".join(names[:10])

        prompt = f"""
[SYSTEM]
You are an expert in product taxonomy.
You see a list of B-level categories and must assign a concise A-level (top level) group name.
The name MUST be:
- At most 4 words
- Generic (no brand names)
- In English.

[USER]
Subcategories:
{sample_names}

Respond with ONLY the A-level category name."""
        label = cached_llm_label(prompt, max_words=4)
        A_name_map[aid] = label
        print(f"A_id {aid} → {label}")
    return A_name_map

A_name_map = name_A_clusters(df_clean)
df_clean["A_name"] = df_clean["A_id"].map(A_name_map)


⚠ Ollama stderr:
 ⠙ ⠙ ⠸ ⠸ ⠴ 
A_id 0 → Technical Support Services
⚠ Ollama stderr:
 ⠙ ⠙ ⠹ ⠼ 
A_id 1 → Mobile Electronics Accessories
⚠ Ollama stderr:
 ⠋ ⠙ ⠹ 
A_id 2 → External Data Storage
⚠ Ollama stderr:
 ⠋ ⠙ ⠹ ⠸ 
A_id 3 → Security Information Management
⚠ Ollama stderr:
 ⠙ ⠹ ⠸ ⠸ 
A_id 4 → Computer Hardware and Cases
⚠ Ollama stderr:
 ⠋ ⠙ ⠸ ⠼ 
A_id 5 → Computer Peripherals
⚠ Ollama stderr:
 ⠙ ⠙ ⠸ ⠼ 
A_id 6 → Display Technology
A_id 7 → Computer Peripherals
⚠ Ollama stderr:
 ⠋ ⠙ ⠹ ⠸ 
A_id 8 → Product Service Management
⚠ Ollama stderr:
 ⠋ ⠹ ⠹ ⠸ 
A_id 9 → Information Systems
⚠ Ollama stderr:
 ⠋ ⠹ ⠸ ⠸ 
A_id 10 → Display Devices
⚠ Ollama stderr:
 ⠙ ⠙ ⠸ ⠸ 
A_id 11 → Mobile Computing Devices
⚠ Ollama stderr:
 ⠋ ⠙ ⠹ ⠸ 
A_id 12 → Computer Graphics Equipment
⚠ Ollama stderr:
 ⠋ ⠹ ⠹ ⠼ 
A_id 13 → Information Technology
⚠ Ollama stderr:
 ⠋ ⠹ ⠸ ⠼ 
A_id 14 → Laptops
⚠ Ollama stderr:
 ⠙ ⠙ ⠸ 
A_id 15 → Input/Output Devices
⚠ Ollama stderr:
 ⠙ ⠙ ⠸ 
A_id 16 → Computing Equipment Care
⚠ Ollama stderr:
 

In [95]:
# Group by discovered A/B/C names and count products
grouped = (
    df_clean
    .groupby(["A_name", "B_name", "C_name"])
    .size()
    .reset_index(name="num_products")
    .sort_values(["A_name", "B_name", "num_products"], ascending=[True, True, False])
)

def print_discovered_hierarchy(df_g):
    for a in df_g["A_name"].dropna().unique():
        print(f"A: {a}")
        print("=" * 80)
        tempA = df_g[df_g["A_name"] == a]

        for b in tempA["B_name"].dropna().unique():
            print(f"  B: {b}")
            tempB = tempA[tempA["B_name"] == b]

            for _, row in tempB.iterrows():
                c = row["C_name"]
                n = row["num_products"]
                print(f"    C: {c}  ({n} products)")
        print("\n")

print_discovered_hierarchy(grouped)


A: Accessories
  B: Device Accessories
    C: - Phone Cases -  (146 products)


A: Cable Communication
  B: Network Cable Products
    C: Network Cables and Adapters  (65 products)
    C: Fiber Optic Cables  (30 products)
    C: Ethernet Cables  (21 products)


A: Computer Accessories
  B: Electronics Peripherals
    C: Computer Components  (37 products)
    C: Display Accessories  (21 products)
  B: Input Devices
    C: Computer Keyboards  (87 products)


A: Computer Components
  B: Computer Hardware
    C: Computer Processors  (77 products)
  B: Computer Memory
    C: Server RAM Components  (30 products)
    C: PC Server RAM  (25 products)


A: Computer Graphics Equipment
  B: Computer Graphics Hardware
    C: Graphics Processors  (33 products)


A: Computer Hardware
  B: Laptop Computer Components
    C: Computer Components  (54 products)
    C: Acer Laptop Accessories  (29 products)
    C: Computer motherboards for laptops  (28 products)


A: Computer Hardware and Cases
  B: Comput

In [99]:
from anytree import Node, RenderTree

# Take unique combinations to avoid duplicate nodes
unique_paths = (
    df_clean[["A_name", "B_name", "C_name"]]
    .dropna()
    .drop_duplicates()
)

# Root node for the discovered taxonomy
root = Node("Computers & Electronics")

# Cache for created nodes to avoid re-creating them
node_cache = {}   # keys like ("A", A_name), ("B", A_name, B_name), ("C", A_name, B_name, C_name)

for _, row in unique_paths.iterrows():
    a = row["A_name"]
    b = row["B_name"]
    c = row["C_name"]

    # A level
    a_key = ("A", a)
    if a_key not in node_cache:
        node_cache[a_key] = Node(a, parent=root)

    a_node = node_cache[a_key]

    # B level
    b_key = ("B", a, b)
    if b_key not in node_cache:
        node_cache[b_key] = Node(b, parent=a_node)

    b_node = node_cache[b_key]

    # C level
    c_key = ("C", a, b, c)
    if c_key not in node_cache:
        node_cache[c_key] = Node(c, parent=b_node)

# Pretty-print the tree
for pre, _, node in RenderTree(root):
    print(f"{pre}{node.name}")


Computers & Electronics
├── Laptops
│   ├── Portable Personal Computers
│   │   ├── Laptops
│   │   └── Laptop
│   └── Portable Computers
│       └── laptops
├── Input Devices for Mobile
│   └── Mobile Input Devices
│       ├── Portable Keyboards
│       └── Laptop Top Cover Keyboards
├── Cable Communication
│   └── Network Cable Products
│       ├── Fiber Optic Cables
│       ├── Ethernet Cables
│       └── Network Cables and Adapters
├── Computing Equipment
│   └── Personal Computing Devices
│       ├── Laptop/Desktop Computers
│       ├── Portable Laptops
│       ├── Desktop Computers
│       └── Laptop Computers
├── External Data Storage
│   ├── Storage Devices
│   │   ├── Computer Server Hardware
│   │   ├── Hard Disk Drives
│   │   └── External Hard Drives and
│   └── External Storage Devices
│       └── HDD Hard Drives
├── Computer Peripherals
│   ├── Computer Input Devices
│   │   ├── ThinkPad Keyboard Sleeves and
│   │   ├── Computer Keyboards
│   │   ├── Acer Keyboards
│   │ 

In [100]:
# Build clean A/B/C summary
abc_df = (
    df_clean
    .groupby(["A_name", "B_name", "C_name"])
    .size()
    .reset_index(name="num_products")
    .sort_values(["A_name", "B_name", "num_products"], ascending=[True, True, False])
)

abc_df


,A_name,B_name,C_name,num_products
0,Accessories,Device Accessories,- Phone Cases -,146
3,Cable Communication,Network Cable Products,Network Cables and Adapters,65
2,Cable Communication,Network Cable Products,Fiber Optic Cables,30
1,Cable Communication,Network Cable Products,Ethernet Cables,21
4,Computer Accessories,Electronics Peripherals,Computer Components,37
...,...,...,...,...
71,Security Information Management,Information Protection Systems,Cyber Security Solutions,30
72,Stationery,Office Supplies,Colour Toner Cartridges,361
73,Technical Support Services,IT Hardware Maintenance,Computer Hardware Support Services,24
74,Technical Support Services,Post-Warranty Maintenance Plans,Extended Warranty Programs,49


In [101]:
df_clean["A_name"]
df_clean["B_name"]
df_clean["C_name"]


1072689                                Laptops
906402                      Portable Keyboards
411281                      Fiber Optic Cables
1047582               Laptop/Desktop Computers
904910                      Portable Keyboards
                          ...                 
231002                          Graphics Cards
978436                  Laptop Display Screens
1101608    Desktops Notebooks Gaming Pavilions
66861                    Tablets and E-readers
1121736                    - Stand Alone Racks
Name: C_name, Length: 4382, dtype: object

In [102]:
# Unique A levels
A_levels = sorted(abc_df["A_name"].unique())
print("A-level categories:")
for a in A_levels:
    print(" -", a)

# Unique B levels
B_levels = sorted(abc_df["B_name"].unique())
print("\nB-level categories:")
for b in B_levels:
    print(" -", b)

# Unique C levels
C_levels = sorted(abc_df["C_name"].unique())
print("\nC-level categories:")
for c in C_levels:
    print(" -", c)


A-level categories:
 - Accessories
 - Cable Communication
 - Computer Accessories
 - Computer Components
 - Computer Graphics Equipment
 - Computer Hardware
 - Computer Hardware and Cases
 - Computer Monitors
 - Computer Peripherals
 - Computing Devices
 - Computing Equipment
 - Computing Equipment Care
 - Cybersecurity Measures
 - Desk Organizers
 - Display Devices
 - Display Technology
 - Electrical Components
 - Electronics
 - Electronics Display
 - External Data Storage
 - External Hard Drives
 - Handheld Electronic Devices
 - Hardware
 - Headphones
 - Information Systems
 - Information Technology
 - Input Devices for Mobile
 - Input/Output Devices
 - Laptops
 - Maintenance Services
 - Mobile Computing Devices
 - Mobile Electronics Accessories
 - Monitor Accessories
 - Network Security Systems
 - Product Service Management
 - Security Information Management
 - Stationery
 - Technical Support Services

B-level categories:
 - Audio Headgear
 - Business Cyber Security Solutions
 - Com

In [103]:
print("A count:", abc_df["A_name"].nunique())
print("B count:", abc_df["B_name"].nunique())
print("C count:", abc_df["C_name"].nunique())


A count: 38
B count: 47
C count: 67


In [104]:
grouped = (
    df_clean
    .groupby(["A_name", "B_name", "C_name"])
    .size()
    .reset_index(name="num_products")
    .sort_values(["A_name", "B_name", "num_products"], ascending=[True, True, False])
)


In [105]:
import re

def normalize_label(s):
    if not isinstance(s, str):
        return s
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    # basic title-case
    s = s.title()
    return s

df_clean["A_name_norm"] = df_clean["A_name"].apply(normalize_label)
df_clean["B_name_norm"] = df_clean["B_name"].apply(normalize_label)
df_clean["C_name_norm"] = df_clean["C_name"].apply(normalize_label)

grouped_norm = (
    df_clean
    .groupby(["A_name_norm", "B_name_norm", "C_name_norm"])
    .size()
    .reset_index(name="num_products")
    .sort_values(["A_name_norm", "B_name_norm", "num_products"], ascending=[True, True, False])
)


In [106]:
grouped_norm.to_csv("discovered_taxonomy_ABC_clean.csv", index=False)


In [107]:
df_clean["A_super"] = "Computers & Electronics"
from anytree import Node, RenderTree

# Select only unique combinations to avoid duplicate nodes
unique_paths = (
    df_clean[["A_super", "A_name", "B_name", "C_name"]]
    .dropna()
    .drop_duplicates()
)

# Create super-root
root = Node("Computers & Electronics")

# Cache to avoid duplicate nodes
node_cache = {}   # keys: ("A_super"), ("A_name", A_name), ("B_name", A_name, B_name), ...

for _, row in unique_paths.iterrows():
    a_super = row["A_super"]                # always Computers & Electronics
    a = row["A_name"]                       # discovered A
    b = row["B_name"]                       # discovered B
    c = row["C_name"]                       # discovered C

    # A_super level (root)
    super_key = ("A_super", a_super)
    if super_key not in node_cache:
        node_cache[super_key] = Node(a_super)
    super_node = node_cache[super_key]

    # A_name level
    a_key = ("A_name", a_super, a)
    if a_key not in node_cache:
        node_cache[a_key] = Node(a, parent=super_node)
    a_node = node_cache[a_key]

    # B_name level
    b_key = ("B_name", a, b)
    if b_key not in node_cache:
        node_cache[b_key] = Node(b, parent=a_node)
    b_node = node_cache[b_key]

    # C_name level
    c_key = ("C_name", a, b, c)
    if c_key not in node_cache:
        node_cache[c_key] = Node(c, parent=b_node)

# Print the hierarchy
for pre, _, node in RenderTree(super_node):
    print(f"{pre}{node.name}")


Computers & Electronics
├── Laptops
│   ├── Portable Personal Computers
│   │   ├── Laptops
│   │   └── Laptop
│   └── Portable Computers
│       └── laptops
├── Input Devices for Mobile
│   └── Mobile Input Devices
│       ├── Portable Keyboards
│       └── Laptop Top Cover Keyboards
├── Cable Communication
│   └── Network Cable Products
│       ├── Fiber Optic Cables
│       ├── Ethernet Cables
│       └── Network Cables and Adapters
├── Computing Equipment
│   └── Personal Computing Devices
│       ├── Laptop/Desktop Computers
│       ├── Portable Laptops
│       ├── Desktop Computers
│       └── Laptop Computers
├── External Data Storage
│   ├── Storage Devices
│   │   ├── Computer Server Hardware
│   │   ├── Hard Disk Drives
│   │   └── External Hard Drives and
│   └── External Storage Devices
│       └── HDD Hard Drives
├── Computer Peripherals
│   ├── Computer Input Devices
│   │   ├── ThinkPad Keyboard Sleeves and
│   │   ├── Computer Keyboards
│   │   ├── Acer Keyboards
│   │ 

In [109]:
# Build a clean summary A/B/C table with product counts
taxonomy_summary = (
    df_clean
    .groupby(["A_name", "B_name", "C_name"])
    .size()
    .reset_index(name="num_products")
    .sort_values(["A_name", "B_name", "num_products"], ascending=[True, True, False])
)

taxonomy_summary.head(20)


,A_name,B_name,C_name,num_products
0,Accessories,Device Accessories,- Phone Cases -,146
3,Cable Communication,Network Cable Products,Network Cables and Adapters,65
2,Cable Communication,Network Cable Products,Fiber Optic Cables,30
1,Cable Communication,Network Cable Products,Ethernet Cables,21
4,Computer Accessories,Electronics Peripherals,Computer Components,37
5,Computer Accessories,Electronics Peripherals,Display Accessories,21
6,Computer Accessories,Input Devices,Computer Keyboards,87
7,Computer Components,Computer Hardware,Computer Processors,77
9,Computer Components,Computer Memory,Server RAM Components,30
8,Computer Components,Computer Memory,PC Server RAM,25
